# Aula 04 — Lógica Proposicional, Conectivos e Permissivos

Este notebook implementa os blocos combinacionais de permissivos da máquina pneumática de envasamento de copos. A implementação separa **solicitação**, **permissivo de partida**, **liberação do comando** e **regra de validação**.

Para cada operação $X$:

$$LIB_X = REQ_X \land M_{valido} \land P_X$$

A implicação $LIB_X \rightarrow P_X$ é usada apenas para verificar segurança; ela não calcula a saída.

## Modelo adotado

- `Auto XOR Manual` garante seleção exclusiva de modo.
- `k1 AND NOT e1 AND NOT b2 AND NOT b3` forma a habilitação geral.
- Uma incoerência entre fins de curso de avanço e recuo bloqueia novas partidas.
- Os resultados calculados são liberações de **início de subetapa**. A sustentação temporal das válvulas e motores pertence à FSM/Grafcet.
- A emergência nunca possui bypass neste módulo.
- `Auto`, `Manual` e `REQ_X` são sinais internos: a FSM gera solicitações automáticas e a IHM gera solicitações manuais autorizadas.

In [1]:
from typing import Dict, List, Tuple, Any

def NOT(a: bool) -> bool:
    return not a

def AND(*args: bool) -> bool:
    return all(args)

def OR(*args: bool) -> bool:
    return any(args)

def XOR(a: bool, b: bool) -> bool:
    return bool(a) ^ bool(b)

def IMPLIES(a: bool, b: bool) -> bool:
    return (not a) or b

def IFF(a: bool, b: bool) -> bool:
    return bool(a) == bool(b)

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    if not dados:
        return 'Tabela vazia'
    colunas = list(dados[0].keys())
    larguras = {c: max(len(c), *(len(str(linha[c])) for linha in dados)) for c in colunas}
    cabecalho = ' | '.join(f'{c:<{larguras[c]}}' for c in colunas)
    divisor = '-+-'.join('-' * larguras[c] for c in colunas)
    corpo = [' | '.join(f'{str(linha[c]):<{larguras[c]}}' for c in colunas) for linha in dados]
    return '\n'.join([cabecalho, divisor, *corpo])

class ControladorPermissivosEnvasadora:
    PARES_FIM_CURSO: Dict[str, Tuple[str, str]] = {
        'Cilindro A': ('c1a', 'c1r'),
        'Cilindro C': ('c3a', 'c3r'),
        'Cilindro D': ('c5a', 'c5r'),
        'Cilindro E': ('c6a', 'c6r'),
        'Cilindro F': ('c7a', 'c7r'),
        'Cilindro G': ('c8a', 'c8r'),
        'Cilindro H': ('c9a', 'c9r'),
    }

    CAMPOS_ESTADO = {
        'k1', 'e1', 'b2', 'b3', 'p0',
        'c1a', 'c1r', 's2', 'c2', 'c3a', 'c3r', 'c4',
        's3', 'c5a', 'c5r', 'c6a', 'c6r', 'p1',
        's4', 'c7a', 'c7r', 't1',
        's5', 'c8a', 'c8r', 'c9a', 'c9r'
    }

    ACOES = (
        'indexar', 'abrir_bico', 'avancar_dosador',
        'entregar_tampa', 'selar', 'elevar', 'extrair'
    )

    @classmethod
    def validar_estado(cls, st: Dict[str, bool]) -> None:
        ausentes = sorted(cls.CAMPOS_ESTADO - set(st))
        if ausentes:
            raise KeyError(f'Variáveis ausentes no estado: {ausentes}')

    @staticmethod
    def modo_valido(req: Dict[str, bool]) -> bool:
        return XOR(req.get('auto', False), req.get('manual', False))

    @staticmethod
    def habilitacao_geral(st: Dict[str, bool]) -> bool:
        return AND(st['k1'], NOT(st['e1']), NOT(st['b2']), NOT(st['b3']))

    @classmethod
    def incoerencias_sensores(cls, st: Dict[str, bool]) -> List[str]:
        return [
            nome for nome, (avanco, recuo) in cls.PARES_FIM_CURSO.items()
            if st[avanco] and st[recuo]
        ]

    @classmethod
    def base_segura(cls, st: Dict[str, bool]) -> bool:
        return cls.habilitacao_geral(st) and not cls.incoerencias_sensores(st)

    @classmethod
    def permissivo_giro_mesa(cls, st: Dict[str, bool]) -> bool:
        cls.validar_estado(st)
        return AND(
            cls.base_segura(st), st['p0'], st['c1a'], st['c3a'], NOT(st['c4']),
            st['c5r'], st['c6r'], st['c7r'], st['c8r'], st['c9a']
        )

    @classmethod
    def permissivo_abrir_bico(cls, st: Dict[str, bool]) -> bool:
        cls.validar_estado(st)
        return AND(
            cls.base_segura(st), st['p0'], st['s2'], st['c2'],
            st['c3r'], NOT(st['c4'])
        )

    @classmethod
    def permissivo_avancar_dosador(cls, st: Dict[str, bool]) -> bool:
        cls.validar_estado(st)
        return AND(
            cls.base_segura(st), st['p0'], st['s2'], st['c2'],
            st['c3r'], st['c4'], NOT(st['c3a'])
        )

    @classmethod
    def permissivo_giro_tampa(cls, st: Dict[str, bool]) -> bool:
        cls.validar_estado(st)
        return AND(
            cls.base_segura(st), st['p0'], st['s3'], st['p1'],
            st['c5r'], st['c6r']
        )

    @classmethod
    def permissivo_prensa(cls, st: Dict[str, bool]) -> bool:
        cls.validar_estado(st)
        return AND(cls.base_segura(st), st['p0'], st['s4'], st['t1'], st['c7r'])

    @classmethod
    def intertravamento_continuo_prensa(cls, st: Dict[str, bool]) -> bool:
        cls.validar_estado(st)
        return AND(cls.base_segura(st), st['p0'], st['s4'], st['t1'])

    @classmethod
    def permissivo_elevador(cls, st: Dict[str, bool]) -> bool:
        cls.validar_estado(st)
        return AND(cls.base_segura(st), st['p0'], st['s5'], st['c8r'], st['c9a'])

    @classmethod
    def permissivo_extrator(cls, st: Dict[str, bool]) -> bool:
        cls.validar_estado(st)
        return AND(cls.base_segura(st), st['p0'], st['s5'], st['c8a'], st['c9a'])

    @classmethod
    def permissivos(cls, st: Dict[str, bool]) -> Dict[str, bool]:
        return {
            'indexar': cls.permissivo_giro_mesa(st),
            'abrir_bico': cls.permissivo_abrir_bico(st),
            'avancar_dosador': cls.permissivo_avancar_dosador(st),
            'entregar_tampa': cls.permissivo_giro_tampa(st),
            'selar': cls.permissivo_prensa(st),
            'elevar': cls.permissivo_elevador(st),
            'extrair': cls.permissivo_extrator(st),
        }

    @classmethod
    def liberacoes(cls, st: Dict[str, bool], req: Dict[str, bool]) -> Dict[str, bool]:
        p = cls.permissivos(st)
        modo_ok = cls.modo_valido(req)
        return {acao: AND(req.get(acao, False), modo_ok, p[acao]) for acao in cls.ACOES}

    @classmethod
    def validar_regras_de_seguranca(cls, st: Dict[str, bool], req: Dict[str, bool]) -> bool:
        p = cls.permissivos(st)
        lib = cls.liberacoes(st, req)
        return all(IMPLIES(lib[acao], p[acao]) for acao in cls.ACOES)

print('[OK] Biblioteca de conectivos e permissivos carregada.')


[OK] Biblioteca de conectivos e permissivos carregada.


In [2]:
from copy import deepcopy
import itertools

def estado_base() -> Dict[str, bool]:
    return {
        'k1': True, 'e1': False, 'b2': False, 'b3': False, 'p0': True,
        'c1a': True, 'c1r': False,
        's2': True, 'c2': True, 'c3a': True, 'c3r': False, 'c4': False,
        's3': True, 'c5a': False, 'c5r': True, 'c6a': False, 'c6r': True, 'p1': True,
        's4': True, 'c7a': False, 'c7r': True, 't1': True,
        's5': True, 'c8a': False, 'c8r': True, 'c9a': True, 'c9r': False,
    }

def requisicao_unica(acao: str, auto: bool = True, manual: bool = False) -> Dict[str, bool]:
    req = {nome: False for nome in ControladorPermissivosEnvasadora.ACOES}
    req.update({'auto': auto, 'manual': manual, acao: True})
    return req

def criar_cenario(nome: str, acao: str, esperado: bool, **alteracoes: bool) -> Dict[str, Any]:
    st = estado_base()
    st.update(alteracoes)
    return {'nome': nome, 'acao': acao, 'esperado': esperado, 'estado': st}

cenarios = [
    criar_cenario('Mesa em condição segura', 'indexar', True),
    criar_cenario('Mesa bloqueada pela prensa avançada', 'indexar', False, c7r=False, c7a=True),
    criar_cenario('Bico com dose aspirada', 'abrir_bico', True, c3a=False, c3r=True),
    criar_cenario('Dosador bloqueado: bico fechado', 'avancar_dosador', False, c3a=False, c3r=True, c4=False),
    criar_cenario('Dosador liberado: bico aberto', 'avancar_dosador', True, c3a=False, c3r=True, c4=True),
    criar_cenario('Tampa bloqueada: sem vácuo', 'entregar_tampa', False, p1=False),
    criar_cenario('Prensa em condição normal', 'selar', True),
    criar_cenario('Prensa bloqueada: temperatura baixa', 'selar', False, t1=False),
    criar_cenario('Extrator liberado: elevador no alto', 'extrair', True, c8r=False, c8a=True),
    criar_cenario('Emergência bloqueia elevador', 'elevar', False, e1=True),
    criar_cenario('Sensores contraditórios da prensa', 'selar', False, c7r=True, c7a=True),
]

relatorio = []
for cenario in cenarios:
    req = requisicao_unica(cenario['acao'])
    controlador = ControladorPermissivosEnvasadora
    liberado = controlador.liberacoes(cenario['estado'], req)[cenario['acao']]
    incoerencias = controlador.incoerencias_sensores(cenario['estado'])
    relatorio.append({
        'Cenário': cenario['nome'],
        'Ação': cenario['acao'],
        'Resultado': 'LIBERADO' if liberado else 'BLOQUEADO',
        'Esperado': 'LIBERADO' if cenario['esperado'] else 'BLOQUEADO',
        'Incoerências': ', '.join(incoerencias) if incoerencias else '-',
    })
    assert liberado is cenario['esperado'], cenario['nome']
    assert controlador.validar_regras_de_seguranca(cenario['estado'], req)

print('=== SIMULAÇÃO DOS PERMISSIVOS DA ENVASADORA ===')
print(formatar_tabela(relatorio))

# 1. Exclusividade dos modos automático e manual.
tabela_modos = []
for auto, manual in itertools.product([False, True], repeat=2):
    valido = XOR(auto, manual)
    tabela_modos.append({'Auto': auto, 'Manual': manual, 'Modo válido': valido})
assert sum(linha['Modo válido'] for linha in tabela_modos) == 2

# 2. A emergência deve bloquear todas as ações, inclusive em modo manual.
estados_validos = {
    'indexar': estado_base(),
    'abrir_bico': criar_cenario('', '', True, c3a=False, c3r=True)['estado'],
    'avancar_dosador': criar_cenario('', '', True, c3a=False, c3r=True, c4=True)['estado'],
    'entregar_tampa': estado_base(),
    'selar': estado_base(),
    'elevar': estado_base(),
    'extrair': criar_cenario('', '', True, c8r=False, c8a=True)['estado'],
}
for acao, st_normal in estados_validos.items():
    req_auto = requisicao_unica(acao, auto=True, manual=False)
    assert ControladorPermissivosEnvasadora.liberacoes(st_normal, req_auto)[acao] is True
    for auto, manual in [(True, False), (False, True)]:
        st_emergencia = deepcopy(st_normal)
        st_emergencia['e1'] = True
        req = requisicao_unica(acao, auto=auto, manual=manual)
        assert not any(ControladorPermissivosEnvasadora.liberacoes(st_emergencia, req).values())

# 3. Teste exaustivo do permissivo da prensa: somente 1 de 64 estados é liberado.
vars_prensa = ['k1', 'e1', 'p0', 's4', 't1', 'c7r']
total_prensa = 0
liberados_prensa = 0
for combo in itertools.product([False, True], repeat=len(vars_prensa)):
    st = estado_base()
    st.update(dict(zip(vars_prensa, combo)))
    st['c7a'] = False
    total_prensa += 1
    liberados_prensa += ControladorPermissivosEnvasadora.permissivo_prensa(st)
assert total_prensa == 64 and liberados_prensa == 1

# 4. Teste exaustivo do permissivo de giro: somente a combinação segura é aceita.
vars_mesa = ['k1', 'e1', 'p0', 'c1a', 'c3a', 'c4', 'c5r', 'c6r', 'c7r', 'c8r', 'c9a']
total_mesa = 0
liberados_mesa = 0
for combo in itertools.product([False, True], repeat=len(vars_mesa)):
    st = estado_base()
    st.update(dict(zip(vars_mesa, combo)))
    st.update({'c1r': False, 'c3r': False, 'c5a': False, 'c6a': False, 'c7a': False, 'c8a': False, 'c9r': False})
    total_mesa += 1
    liberados_mesa += ControladorPermissivosEnvasadora.permissivo_giro_mesa(st)
assert total_mesa == 2048 and liberados_mesa == 1

resumo_testes = [
    {'Teste': 'Cenários operacionais', 'Resultado': f'{len(cenarios)}/{len(cenarios)} aprovados'},
    {'Teste': 'Exclusividade Auto XOR Manual', 'Resultado': '4 combinações verificadas'},
    {'Teste': 'Emergência sem bypass', 'Resultado': '7 ações x 2 modos bloqueadas'},
    {'Teste': 'Permissivo da prensa', 'Resultado': f'{liberados_prensa}/{total_prensa} estado liberado'},
    {'Teste': 'Permissivo da mesa', 'Resultado': f'{liberados_mesa}/{total_mesa} estado liberado'},
]
print('\n=== RELATÓRIO DE TESTES AUTOMÁTICOS ===')
print(formatar_tabela(resumo_testes))
print('\n[OK] Todos os permissivos, intertravamentos e invariantes foram validados.')


=== SIMULAÇÃO DOS PERMISSIVOS DA ENVASADORA ===
Cenário                             | Ação            | Resultado | Esperado  | Incoerências
------------------------------------+-----------------+-----------+-----------+-------------
Mesa em condição segura             | indexar         | LIBERADO  | LIBERADO  | -           
Mesa bloqueada pela prensa avançada | indexar         | BLOQUEADO | BLOQUEADO | -           
Bico com dose aspirada              | abrir_bico      | LIBERADO  | LIBERADO  | -           
Dosador bloqueado: bico fechado     | avancar_dosador | BLOQUEADO | BLOQUEADO | -           
Dosador liberado: bico aberto       | avancar_dosador | LIBERADO  | LIBERADO  | -           
Tampa bloqueada: sem vácuo          | entregar_tampa  | BLOQUEADO | BLOQUEADO | -           
Prensa em condição normal           | selar           | LIBERADO  | LIBERADO  | -           
Prensa bloqueada: temperatura baixa | selar           | BLOQUEADO | BLOQUEADO | -           
Extrator liberado: ele

## Conclusão

O módulo bloqueia novas partidas diante de emergência, modo inconsistente, ausência de permissivos ou sensores contraditórios. Os testes exaustivos demonstram que a prensa e a mesa são liberadas somente na única combinação integralmente segura de seus respectivos subconjuntos de variáveis.

A lógica não substitui o circuito físico de emergência nem o sequenciador temporal da máquina. Ela constitui a camada combinacional que será consumida pela FSM global e pelas FSMs das cinco estações.